 Make sure you have the datasets, transformer and torch libraries

In [ ]:
# install the necessary libraries
%pip install datasets
%pip install transformer
%pip install torch


 Now we import the libraries

In [ ]:
# import the libraries and functions we need
import transformers, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import datasets as ds



In [ ]:

#initialize one variables including the name of the model we want to use
#the tokenizer we want to use and the actual model
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


 Make the Model run on the GPU

In [ ]:

#set the device to cuda if the GPU is available, else set it to cpu
#then tell the model to use that device
if torch.cuda.is_available():
    device = "cuda"
else: device = "cpu"

print(device)
model = model.to(device)


In [ ]:
# load the training and testing jsonl data sets  and turn them into data sets from datasetDict's
dataset = ds.load_dataset("json", data_files="shakespearTrain.jsonl",encoding='utf-8')
testDataset = ds.load_dataset("json", data_files="shakespearTest.jsonl",encoding='utf-8')


if isinstance(dataset, ds.DatasetDict):
    # Concatenate all splits into a single Dataset
    trainDataset = ds.concatenate_datasets([dataset[split] for split in dataset])
else:
    raise ValueError("Loaded data is not a DatasetDict")

if isinstance(testDataset, ds.DatasetDict):
    # Concatenate all splits into a single Dataset
    testDataset = ds.concatenate_datasets([testDataset[split] for split in testDataset])
else:
    raise ValueError("Loaded data is not a DatasetDict")
#


In [ ]:
#preproceesss the data from the jsonl file
def preprocess(dataset):
    #seperate the different types of data we are giving it,
    #training data that is seperated into prompts and responses
    inputs = dataset["prompt"]
    responses = dataset["response"]

    # Convert inputs and responses to lists of strings
    inputs = [str(i) for i in inputs]
    responses = [str(r) for r in responses]


    # Tokenize the articles (inputs) with padding and truncation to a max length of 512
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True, return_tensors="pt")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(responses, max_length=128, padding="max_length", truncation=True, return_tensors="pt")

    model_inputs["labels"] = labels["input_ids"]

    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

    return model_inputs

In [ ]:
#map the data to a useable format for the trainer
tokenized_train_dataset = trainDataset.map(preprocess, batched=True)
print("done")

tokenized_eval_dataset = testDataset.map(preprocess, batched=True)


In [ ]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir='./results',              # Directory to save the model checkpoints
    eval_strategy="epoch",         # Evaluate the model at the end of every epoch
    learning_rate=2e-5,                  # Learning rate for the optimizer
    per_device_train_batch_size=8,       # Batch size for training
    per_device_eval_batch_size=8,        # Batch size for evaluation
    weight_decay=0.01,                   # Regularization to prevent overfitting
    save_total_limit=3,                  # Only keep the last 3 checkpoints
    num_train_epochs=4,                  # Number of training epochs
    predict_with_generate=True,          # Enable text generation during evaluation
    logging_dir="./logs",  # Directory for storing training logs
    report_to="none"
)


In [ ]:
# Create the trainer object
trainer = Seq2SeqTrainer(
    model=model,                         # The model to be trained
    args=training_args,                  # The training arguments defined earlier
    train_dataset=tokenized_train_dataset,  # The tokenized training dataset
    eval_dataset=tokenized_eval_dataset,    # The tokenized evaluation dataset
    tokenizer=tokenizer                  # The tokenizer to handle input and output
)


In [ ]:
#TRAINN!!!! and wait :(
trainer.train()


In [ ]:
# Evaluate the model on the testing dataset
metrics = trainer.evaluate()

# Print the evaluation metrics
print(metrics)


In [ ]:
# save the model
REPO_NAME = "JaxsonYorke/EnglishToShakespearean"

# save model and tokenizer
model.save_pretrained(REPO_NAME)
tokenizer.save_pretrained(REPO_NAME)


In [ ]:
def askQuestion(text):
  # Tokenize the input text and move it to the correct device
  inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(device)

  # Load the model from the saved REPO_NAME
  model = AutoModelForSeq2SeqLM.from_pretrained(REPO_NAME).to(device)

  # Generate the summary using the fine-tuned model
  response_ids = model.generate(inputs["input_ids"], max_length=2048, num_beams=4, early_stopping=True)

  # Decode the generated summary back into text and return it
  return tokenizer.decode(response_ids[0], skip_special_tokens=True)

In [ ]:
print(askQuestion("Take the winding path to reach the lake and open your book to the first page."))